# Methodology Classifier (v2.3 & v2.4): SPECTER + XGBoost

- This notebook builds a methodology classifier using:
	- SPECTER on Title + Abstract
	- SMOTE oversampling to address class imbalance
	- XGBoost classifier

### 1. Load and Prepare Methodology Dataset

In [ ]:
import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("Data/methodology.csv")   

# Combine Title + Abstract into input_text
df["input_text"] = (
    df["Title"].astype(str).str.strip() + " " +
    df["Abstract"].astype(str).str.strip()
)

# Preview
print(df.shape)
df[["Title","Abstract","Methodology","input_text"]].head()

### 2. Check Methodology Distribution and Encode Labels

In [ ]:
from collections import Counter
from sklearn.preprocessing import LabelEncoder

# Check class counts
print("Methodology distribution:")
for m,c in Counter(df["Methodology"]).items():
    print(f"  {m}: {c}")

# Encode labels
le = LabelEncoder()
df["Method_label"] = le.fit_transform(df["Methodology"])

# Preview label mapping
df[["Methodology","Method_label"]].drop_duplicates().sort_values("Method_label")

#### 3. Generate SPECTER Embeddings  
Load the `allena-specter` model to turn each `input_text` into a 768-dim dense vector.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the pretrained SPECTER model
model = SentenceTransformer("allenai-specter")

# Encode all input_text (this can take ~1–2 min)
X = model.encode(
    df["input_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

# Your label array stays the same
y = df["Method_label"].values

print(f"SPECTER embeddings shape: {X.shape}")

#### 4. Train-Test Split  
80/20 stratified split on your new embeddings.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

### 5. Train XGBoost Classifier (v2.3 Baseline)

In [ ]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 5.1 Balance the training set
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print(f"Resampled train size: {X_res.shape}, Class counts: {pd.Series(y_res).value_counts().to_dict()}")

# 5.2 Train XGBoost on the resampled data
xgb = XGBClassifier(
    use_label_encoder=False,   # suppress deprecation warning
    eval_metric="mlogloss",    # multiclass log-loss
    random_state=42
)
xgb.fit(X_res, y_res)

print("✅ Baseline XGBoost model trained")

### 6. Evaluate Model Performance


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set
y_pred = xgb.predict(X_test)

# Decode numeric labels back to methodology names
method_names = le.inverse_transform([0,1,2])  # [“Mixed”, “Qualitative”, “Quantitative”]

# Print detailed metrics
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=method_names))

# Show confusion matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

### 7. Save Model and encoder

In [ ]:
import joblib, os

joblib.dump(le,"Artefacts/methodology_label_encoder_v2.3.pkl")
joblib.dump(xgb,"Artefacts/methodology_xgb_v2.3.pkl")

print("✅ Saved LabelEncoder and XGBoost model to Artefacts/")


### Classification Results Summary (v2.3 – SPECTER + XGBoost)

The classifier achieved the following evaluation metrics on the test set:

- **Accuracy:** 0.75  
- **Macro F1-score:** 0.66  
- **Weighted F1-score:** 0.74  

🔍 **Per-Class F1 Scores:**

| Methodology  | F1 Score | Comments                                            |
| ------------ | -------- | --------------------------------------------------- |
| Qualitative  | 0.83     | Strong precision & recall; clear linguistic cues.   |
| Quantitative | 0.81     | Good overall; minor confusion with Mixed abstracts. |
| Mixed        | 0.35     | Under-represented; consider more oversampling or CW.|

### Confusion Matrix (Heatmap)
We plot the confusion matrix as a heatmap to visualize where the model is confusing between subfields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – Methodology Classifier (v2.3)')
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning (XGBoost) - v2.4

We use `GridSearchCV` to optimize key hyperparameters:

- `max_depth`: Maximum tree depth (model complexity)
- `learning_rate`: Step size shrinkage
- `n_estimators`: Number of boosting rounds

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200]
}

# Initialize base model (remove deprecated args)
xgb_base = XGBClassifier(eval_metric="mlogloss", random_state=42, use_label_encoder=False)

# Grid Search with 3-fold CV
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    verbose=1,
    n_jobs=1  # FIX: Avoid BrokenProcessPool in Jupyter
)

# Fit to training data
grid_search.fit(X_train, y_train)

# Inspect best parameters and score
print("Best Params:", grid_search.best_params_)
print("Best CV Macro-F1:", grid_search.best_score_)

In [ ]:
# Use best model from GridSearchCV
xgb_tuned = grid_search.best_estimator_
y_pred_tuned = xgb_tuned.predict(X_test)

# Classification Report
print("Classification Report (Tuned XGBoost):\n")
print(classification_report(y_test, y_pred_tuned, target_names=method_names))

# Confusion Matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_tuned))

In [ ]:
import joblib
import os

# Save the tuned XGBoost model for methodology classification
joblib.dump(xgb_tuned, "Artefacts/methodology_xgb_model_v2.4_tuned.pkl")
print("✅ Saved tuned methodology model as v2.4")

## 🔧 Tuning Summary and Result Analysis (v2.4)

We performed hyperparameter tuning using `GridSearchCV` to find the optimal configuration for the XGBoost classifier on SPECTER embeddings.  
The goal was to improve performance on the **Methodology** classification task, especially for the under-represented “Mixed” class.

### **Parameter Grid Used**  
We searched across:  
- `max_depth`: [3, 5, 7] – controls tree depth (model complexity)  
- `learning_rate`: [0.05, 0.1, 0.2] – controls how much each tree contributes  
- `n_estimators`: [100, 200] – number of boosting rounds  

---

### ✅ **Best Parameters Found**  
- `learning_rate`: 0.2  
- `max_depth`: 3  
- `n_estimators`: 100  
- **Best CV Macro-F1 (3-fold):** 0.63  

---

### 📊 **Final Evaluation on Test Set (v2.4 – Tuned XGBoost)**  
- **Accuracy:** 0.73  
- **Macro F1-score:** 0.58  
- **Weighted F1-score:** 0.69  

#### 🔍 **Class-wise F1 Breakdown:**

| Methodology   | F1 Score | Notes                                       |
| ------------- | -------- | ------------------------------------------- |
| Qualitative   | 0.83     | Stable performance, unchanged from v2.3     |
| Quantitative  | 0.79     | Slightly lower than v2.3, still strong      |
| Mixed         | 0.11     | Significantly degraded—tuning hurt minority |

---

## 🧠 Interpretation

- Hyperparameter tuning improved cross-validation macro-F1 but **degraded** the “Mixed” class on the test set.  
- Shallow trees (`max_depth=3`) with higher learning rate (`0.2`) converged quickly but over-regularized the minority class.  
- “Qualitative” and “Quantitative” remained robust, but “Mixed” dropped from 0.35 (v2.3) to 0.11 (v2.4).  

---

## 📝 Next Steps

- **v2.5 Class-Weights Experiment:** Replace SMOTE with XGBoost’s `scale_pos_weight` to rebalance classes without oversampling.  
- **Ensemble Approaches:** Combine SMOTE and class-weights or ensemble XGB with a linear baseline (e.g., Logistic Regression).  
- **Feature Engineering:** Add keyword flags or hybrid embeddings (title-only vs full text).  
- **Interpretability:** Use SHAP to diagnose which features drive “Mixed” predictions.  
- **Data Expansion:** Collect more “Mixed” examples to strengthen the model’s exposure.  

Overall, while v2.4 optimized global metrics, we need a targeted imbalance strategy to recover the under-represented class.  

## v2.5 Experiment: Class Weights Instead of SMOTE

Instead of oversampling with SMOTE, we’ll leverage XGBoost’s `scale_pos_weight` to rebalance classes.  
1. Compute balanced class weights (inverse to training frequencies).  
2. Retrain XGBoost on the original SPECTER embeddings (`X_train`, `y_train`) with these weights.  
3. Evaluate on `X_test`/`y_test` and compare “Mixed” F1 to v2.3 & v2.4.

In [ ]:
# 9.1 Compute class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = dict(zip(classes, weights))
print("Computed class weights:", class_weights)

# 9.2 Train XGBoost with class weights
from xgboost import XGBClassifier

xgb_w = XGBClassifier(
    use_label_encoder=False,
    eval_metric="mlogloss",
    scale_pos_weight=class_weights,  # XGBoost will apply these per-class
    random_state=42
)
xgb_w.fit(X_train, y_train)
print("✅ Trained class-weighted XGBoost (v2.5)")

# 9.3 Evaluate on test set
from sklearn.metrics import classification_report, confusion_matrix

y_pred_w = xgb_w.predict(X_test)
print("=== v2.5 Class-Weighted XGB Classification Report ===\n")
print(classification_report(y_test, y_pred_w, target_names=le.inverse_transform(classes)))

### 10. Save v2.5 Class-Weighted Model Artefact

In [ ]:
import os, joblib

joblib.dump(xgb_w, "Artefacts/methodology_xgb_class_weighted_v2.5.pkl")
print("✅ Saved class-weighted XGBoost model as v2.5")

## 📝 Results Summary (v2.5 – Class-Weighted XGBoost)

After replacing SMOTE with XGBoost’s `scale_pos_weight`, the model achieved on the test set:

- **Accuracy:** 0.74  
- **Macro F1-score:** 0.61  
- **Weighted F1-score:** 0.71  

🔍 **Class-wise F1 Breakdown:**

| Methodology   | F1 Score | Notes                                          |
| ------------- | -------- | ---------------------------------------------- |
| Qualitative   | 0.83     | Same as before; well-represented class         |
| Quantitative  | 0.79     | Stable performance                             |
| Mixed         | 0.20     | Improved over v2.4 (0.11) but below v2.3 (0.35)|

---

## 🧠 Interpretation

- Class weighting boosted **Mixed** F1 from **0.11** (v2.4) to **0.20**, but still below the SMOTE-only v2.3 baseline **0.35**.  
- “Qualitative” and “Quantitative” remain strong, but the under-represented “Mixed” class still lags.  

---

## v2.5a Experiment: Manual Class Weights

Instead of computing balanced weights, we’ll assign **Mixed = 2** and **Qualitative/Quantitative = 1** to further up-weight the under-represented class.

In [ ]:
from xgboost import XGBClassifier
import numpy as np

# Manual weights by name
manual_weights = {
    "Mixed":       2,
    "Qualitative": 1,
    "Quantitative":1
}

# Map methodology names -> numeric labels
name_to_label = dict(zip(le.classes_, le.transform(le.classes_)))

# Build numeric_weights: label -> weight
numeric_weights = {
    name_to_label[name]: weight
    for name, weight in manual_weights.items()
}

# Build sample_weight array
import numpy as np
sample_weight = np.array([ numeric_weights[y] for y in y_train ])

# Fit XGBoost with sample weights
from xgboost import XGBClassifier
xgb_manual = XGBClassifier(
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)
xgb_manual.fit(X_train, y_train, sample_weight=sample_weight)

print("✅ Trained manual‐weighted XGBoost (v2.5a)")

In [ ]:
y_pred_manual = xgb_manual.predict(X_test)
print(classification_report(y_test, y_pred_manual, target_names=le.classes_))

In [ ]:
joblib.dump(xgb_manual, "Artefacts/methodology_xgb_manual_weights_v2.5a.pkl")
print("✅ Saved manual class-weighted XGBoost model as v2.5a")

## 📋 Results Summary (v2.5a – Manual Class Weights)

By setting **Mixed = 2** and **Qualitative/Quantitative = 1** via `sample_weight`:

- **Accuracy:** 0.74  
- **Macro F1-score:** 0.61  
- **Weighted F1-score:** 0.71  

🔍 **Class-wise F1 Breakdown:**

| Methodology   | F1 Score | Notes                                           |
| ------------- | -------- | ----------------------------------------------- |
| Qualitative   | 0.82     | Very similar to SMOTE and automatic weights     |
| Quantitative  | 0.80     | Stable performance                              |
| Mixed         | 0.19     | Improved over v2.4 (0.11) but below v2.3 (0.35) |

---

## 🧠 Interpretation

- Manual up-weighting boosted **Mixed** F1 from **0.11** → **0.19**, but still below SMOTE-only v2.3 (0.35).  
- “Qualitative” and “Quantitative” remain robust, indicating the weight scheme did not harm majority classes.  
- Overall, neither class-weights nor tuning fully recover Mixed performance — suggesting we may need SMOTE + weights together or more data for the Mixed category.

---

## 🚀 Next Steps

1. **v2.6 Ensemble SMOTE & Manual Weights:** Combine predictions from v2.3 (SMOTE) and v2.5a (manual weights) via simple voting or stacking.  
2. **Data Augmentation:** Create more “Mixed” examples (e.g. via back-translation).  
3. **Fine-Tune a Transformer:** Train a BERT/SciBERT model end-to-end on methodology labels.  
4. **Investigate Features:** Add keyword flags or POS-based signals to better distinguish Mixed papers.  